# LMM Standalone + code_mapping + batch=256: Mortality Prediction on MIMIC-III

**Diagnostic run: Is GRASP or LMM the oscillation source?**

This notebook runs **LMM standalone (no GRASP)** with code_mapping, batch_size=256, and all 3 training fixes. If LMM converges smoothly without GRASP's clustering, the oscillation seen in previous runs is caused by k-means instability, not the LMM architecture.

**Paper**: Ali Behrouz et al. "Titans: Learning to Memorize at Test Time." arXiv 2501.00663, 2025.

**Previous results (all with GRASP, batch=32):**
- GRASP+LMM+codemap, lr=5e-4, no fixes:  test roc_auc = 0.553
- GRASP+LMM+codemap, lr=1e-4, no fixes:  test roc_auc = 0.537
- GRASP+LMM+codemap, lr=1e-4, 3 fixes:   test roc_auc = 0.556
- GRASP+GRU+codemap (sweep best):         test roc_auc = 0.639

**What's different in this run:**
- No GRASP clustering — LMM standalone
- batch_size=256 (vs 32) — 8x larger batches for GPU utilization
- 3 fixes: lr=1e-4, weight_decay=1e-4 (correct), max_grad_norm=1.0

## Step 1: Load the MIMIC-III Dataset

We load the MIMIC-III dataset using PyHealth's `MIMIC3Dataset` class.

**Two configurations:**
- **Local (default):** Set `dev=True` with synthetic GCS data for quick testing
- **H200 / full run:** Set `USE_LOCAL_DATA = True` below and point `LOCAL_ROOT` to your MIMIC-III CSV directory

In [ ]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset

# ── Configuration ─────────────────────────────────────────
# Set USE_LOCAL_DATA = True on the H200 where MIMIC-III CSVs live
USE_LOCAL_DATA = True
LOCAL_ROOT = "/home/lolowo2"           # directory with ADMISSIONS.csv.gz, etc.
LOCAL_CACHE = "/home/lolowo2/tmp/pyhealth_cache"
# ──────────────────────────────────────────────────────────

if USE_LOCAL_DATA:
    base_dataset = MIMIC3Dataset(
        root=LOCAL_ROOT,
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=LOCAL_CACHE,
        dev=False,  # full dataset
    )
else:
    base_dataset = MIMIC3Dataset(
        root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=tempfile.TemporaryDirectory().name,
        dev=True,  # small synthetic subset
    )

base_dataset.stats()

## Step 2: Define the Mortality Prediction Task (with code_mapping)

The `MortalityPredictionMIMIC3` task with `code_mapping` enabled:
- **conditions**: ICD-9-CM → CCS-CM (~530 groups)
- **procedures**: ICD-9-PROC → CCS-PROC (~230 groups)
- **drugs**: NDC → ATC (~6,000 groups)

This collapses the vocabulary so each embedding gets trained on enough data to be meaningful.

In [ ]:
from pyhealth.tasks import MortalityPredictionMIMIC3

# Enable code_mapping to collapse granular codes into grouped vocabularies
task = MortalityPredictionMIMIC3(
    code_mapping={
        "conditions": ("ICD9CM", "CCSCM"),
        "procedures": ("ICD9PROC", "CCSPROC"),
        "drugs": ("NDC", "ATC"),
    }
)

samples = base_dataset.set_task(task)

print(f"Generated {len(samples)} samples")
print(f"\nInput schema: {samples.input_schema}")
print(f"Output schema: {samples.output_schema}")

## Step 3: Split Data and Create Loaders

We split by patient to avoid data leakage, then create batched data loaders.

In [ ]:
from pyhealth.datasets import split_by_patient, get_dataloader

train_dataset, val_dataset, test_dataset = split_by_patient(
    samples, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=256, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=256, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=256, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: 256")

## Compute Tracking Setup

We log hardware specs, energy consumption, GPU power/utilization, and CO2 emissions.

Install dependencies if needed:
```bash
pip install codecarbon pynvml
```

**Metrics tracked per training run:**

| Metric | What it measures | Unit |
|---|---|---|
| Wall time | Real elapsed clock time | seconds |
| Energy consumed | Electricity drawn by GPU | kWh |
| CO2 emissions | Energy × grid carbon intensity | kg CO2eq |
| Peak / avg GPU power | How hard the chip works | Watts |
| GPU utilization | % time GPU cores are computing | % |
| GPU memory | VRAM allocated vs available | GB |
| Energy per epoch | Diminishing returns indicator | kWh |
| Energy per sample | Unit cost for deployment | Joules |
| Batch throughput | Samples processed per second | samples/s |

In [ ]:
import time
import platform
import torch

# ── Hardware Info ──────────────────────────────────────────
print("=" * 60)
print("Hardware Information")
print("=" * 60)
print(f"Platform:     {platform.system()} {platform.machine()}")
print(f"Python:       {platform.python_version()}")
print(f"PyTorch:      {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:          {gpu_name}")
    print(f"GPU Memory:   {gpu_mem_gb:.1f} GB")
    print(f"CUDA:         {torch.version.cuda}")
else:
    print("GPU:          None (CPU only)")
    print("  Note: Compute metrics will be limited without GPU")

# ── GPU Monitoring Helper ─────────────────────────────────
gpu_available = torch.cuda.is_available()
nvml_available = False

if gpu_available:
    try:
        import pynvml
        pynvml.nvmlInit()
        nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        nvml_available = True
        print(f"\npynvml:       initialized (live GPU monitoring enabled)")
    except (ImportError, Exception) as e:
        print(f"\npynvml:       not available ({e})")
        print("  Install with: pip install pynvml")

# ── Carbon Tracking Helper ────────────────────────────────
codecarbon_available = False
try:
    from codecarbon import EmissionsTracker
    codecarbon_available = True
    print(f"codecarbon:   available (CO2 tracking enabled)")
except ImportError:
    print(f"codecarbon:   not available")
    print("  Install with: pip install codecarbon")


class ComputeTracker:
    """Tracks energy, GPU metrics, and timing for a training run."""

    def __init__(self, name, num_samples, num_epochs):
        self.name = name
        self.num_samples = num_samples
        self.num_epochs = num_epochs
        self.power_readings = []
        self.util_readings = []
        self.mem_readings = []

    def start(self):
        self.start_time = time.time()
        if gpu_available:
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        if codecarbon_available:
            self.tracker = EmissionsTracker(
                project_name=self.name,
                log_level="error",
                save_to_file=False,
            )
            self.tracker.start()
        self._sample_gpu()
        return self

    def _sample_gpu(self):
        if nvml_available:
            power = pynvml.nvmlDeviceGetPowerUsage(nvml_handle) / 1000  # mW -> W
            util = pynvml.nvmlDeviceGetUtilizationRates(nvml_handle).gpu
            mem_info = pynvml.nvmlDeviceGetMemoryInfo(nvml_handle)
            self.power_readings.append(power)
            self.util_readings.append(util)
            self.mem_readings.append(mem_info.used / 1e9)

    def sample(self):
        """Call periodically during training to collect GPU readings."""
        self._sample_gpu()

    def stop(self):
        if gpu_available:
            torch.cuda.synchronize()
        self.wall_time = time.time() - self.start_time
        self._sample_gpu()

        self.emissions_data = None
        if codecarbon_available:
            self.tracker.stop()
            self.emissions_data = self.tracker.final_emissions_data

        return self.report()

    def report(self):
        r = {"name": self.name}
        r["wall_time_s"] = self.wall_time
        r["wall_time_min"] = self.wall_time / 60
        r["num_epochs"] = self.num_epochs
        r["num_samples"] = self.num_samples

        # Throughput
        total_samples_processed = self.num_samples * self.num_epochs
        r["batch_throughput_sps"] = total_samples_processed / self.wall_time
        r["time_per_epoch_s"] = self.wall_time / max(self.num_epochs, 1)

        # Energy (from codecarbon)
        if self.emissions_data:
            r["energy_kwh"] = self.emissions_data.energy_consumed
            r["co2_kg"] = self.emissions_data.emissions
            r["energy_per_epoch_kwh"] = r["energy_kwh"] / max(self.num_epochs, 1)
            r["energy_per_sample_j"] = (r["energy_kwh"] * 3.6e6) / max(total_samples_processed, 1)
        else:
            r["energy_kwh"] = None
            r["co2_kg"] = None

        # GPU metrics (from pynvml)
        if self.power_readings:
            r["peak_power_w"] = max(self.power_readings)
            r["avg_power_w"] = sum(self.power_readings) / len(self.power_readings)
            r["avg_gpu_util_pct"] = sum(self.util_readings) / len(self.util_readings)
            r["peak_gpu_mem_gb"] = max(self.mem_readings)
        elif gpu_available:
            r["peak_gpu_mem_gb"] = torch.cuda.max_memory_allocated() / 1e9

        return r


def print_report(r):
    print(f"\n{'=' * 50}")
    print(f"Compute Report: {r['name']}")
    print(f"{'=' * 50}")
    print(f"Wall time:          {r['wall_time_s']:.1f}s ({r['wall_time_min']:.2f} min)")
    print(f"Epochs:             {r['num_epochs']}")
    print(f"Time per epoch:     {r['time_per_epoch_s']:.2f}s")
    print(f"Throughput:         {r['batch_throughput_sps']:.1f} samples/s")

    if r.get("energy_kwh") is not None:
        print(f"\nEnergy consumed:    {r['energy_kwh']:.6f} kWh")
        print(f"CO2 emissions:      {r['co2_kg']:.6f} kg CO2eq")
        print(f"Energy per epoch:   {r['energy_per_epoch_kwh']:.6f} kWh")
        print(f"Energy per sample:  {r['energy_per_sample_j']:.4f} J")

    if r.get("peak_power_w"):
        print(f"\nPeak GPU power:     {r['peak_power_w']:.0f} W")
        print(f"Avg GPU power:      {r['avg_power_w']:.0f} W")
        print(f"Avg GPU utilization: {r['avg_gpu_util_pct']:.0f}%")

    if r.get("peak_gpu_mem_gb"):
        total_gb = gpu_mem_gb if gpu_available else 0
        used = r["peak_gpu_mem_gb"]
        print(f"Peak GPU memory:    {used:.2f} GB / {total_gb:.0f} GB ({used/max(total_gb,1)*100:.0f}%)")

    print(f"{'=' * 50}")


print("\nComputeTracker ready.")

## Step 4: LMM Standalone + code_mapping (no GRASP)

LMM as a direct sequence encoder — no clustering, no GCN, no gating. Each patient is processed independently using surprise-based memory.

If this converges smoothly (no ±0.06 oscillation), GRASP's k-means is the instability source.

In [ ]:
from pyhealth.models import LMM

NUM_EPOCHS = 50

lmm_model = LMM(
    dataset=samples,
    embedding_dim=32,
    hidden_dim=32,
    memory_depth=2,
)

print(f"LMM standalone model: {sum(p.numel() for p in lmm_model.parameters()):,} parameters")

In [ ]:
from pyhealth.trainer import Trainer

lmm_trainer = Trainer(
    model=lmm_model,
    metrics=["roc_auc", "pr_auc", "accuracy", "f1"],
)

lmm_tracker = ComputeTracker(
    "LMM-standalone+codemap+batch256", len(train_dataset), NUM_EPOCHS,
).start()

lmm_trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=NUM_EPOCHS,
    monitor="roc_auc",
    optimizer_params={"lr": 1e-4},
    weight_decay=1e-4,
    max_grad_norm=1.0,
)

lmm_compute = lmm_tracker.stop()
print_report(lmm_compute)

## Step 5: Results

GRASP+LMM+code_mapping performance and compute cost.

In [ ]:
lmm_results = lmm_trainer.evaluate(test_dataloader)

print("=" * 60)
print("LMM Standalone + code_mapping + batch=256 — Test Set")
print("=" * 60)
for metric, value in lmm_results.items():
    print(f"  {metric}: {value:.4f}")

print("\n--- Reference (all GRASP+LMM, batch=32) ---")
print("  GRASP+LMM+codemap, 3 fixes:   test roc_auc = 0.556")
print("  GRASP+LMM+codemap, no fixes:   test roc_auc = 0.553")
print("  GRASP+GRU+codemap (sweep):     test roc_auc = 0.639")

## Step 7: Extract Patient Embeddings

Both models produce patient embeddings that can be used for downstream tasks like patient similarity search or cohort discovery.

In [ ]:
import torch

lmm_model.eval()
test_batch = next(iter(test_dataloader))
test_batch["embed"] = True

with torch.no_grad():
    output = lmm_model(**test_batch)

print(f"Embedding shape: {output['embed'].shape}")
print(f"  - Batch size: {output['embed'].shape[0]}")
print(f"  - Embedding dim: {output['embed'].shape[1]}")

print("\nSample Predictions:")
predictions = output["y_prob"].cpu().numpy()
true_labels = output["y_true"].cpu().numpy()

for i in range(min(5, len(predictions))):
    pred = predictions[i][0]
    true = int(true_labels[i][0])
    label = "Mortality" if pred > 0.5 else "Survival"
    print(f"  Patient {i + 1}: Predicted={pred:.3f}, True={true}, -> {label}")

## Step 8: Compute Cost Summary for Policy

This section translates raw compute metrics into policy-relevant context. The goal is to answer: **what is the real environmental cost of this ML research?**

In [ ]:
from datetime import datetime

total_energy = lmm_compute.get("energy_kwh") or 0
total_co2 = lmm_compute.get("co2_kg") or 0
total_time = lmm_compute["wall_time_s"]

print("=" * 60)
print("ENVIRONMENTAL IMPACT — POLICY SUMMARY")
print("=" * 60)
print(f"\n--- LMM standalone + codemap + batch=256 footprint ---")
print(f"Wall time:            {total_time:.0f}s ({total_time/60:.1f} min)")

if total_energy > 0:
    lightbulb_min = (total_energy / 0.01) * 60
    home_fraction = total_energy / 30 * 100
    miles_driven = total_co2 / 0.4

    print(f"Energy consumed:      {total_energy:.6f} kWh")
    print(f"CO2 emissions:        {total_co2:.6f} kg CO2eq")
    print(f"\n--- In everyday terms ---")
    print(f"10W LED bulb for:     {lightbulb_min:.0f} minutes")
    print(f"US home daily usage:  {home_fraction:.4f}%")
    print(f"Equivalent driving:   {miles_driven:.3f} miles")

if lmm_compute.get("avg_gpu_util_pct") is not None:
    util = lmm_compute["avg_gpu_util_pct"]
    mem = lmm_compute.get("peak_gpu_mem_gb", 0)
    total_gb = gpu_mem_gb if gpu_available else 80
    print(f"\n--- GPU efficiency ---")
    print(f"Avg GPU utilization: {util:.0f}%")
    print(f"Memory:              {mem:.1f} GB / {total_gb:.0f} GB ({mem/total_gb*100:.0f}%)")
    if mem < total_gb * 0.1:
        print(f"  -> Could run on < {max(4, mem*2):.0f} GB GPU (consumer card)")

print("=" * 60)

## Step 9: Loss Landscape Visualization

Visualizes the 3D loss surface around the trained model's weights using filter-normalized random directions (Li et al., NeurIPS 2018). Shows whether the model landed in a wide stable basin (good) or a sharp narrow basin (oscillation risk).

**Three fixes applied to this run:**
- `weight_decay=1e-4` as own Trainer argument (correctly applied)
- `max_grad_norm=1.0` outer gradient clipping
- `lr=1e-4` learning rate

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt


def random_direction(state_dict):
    """Filter-normalized random direction in weight space."""
    direction = {}
    for k, v in state_dict.items():
        if v.dtype in (torch.float32, torch.float16, torch.bfloat16):
            d = torch.randn_like(v)
            w_norm = v.norm()
            d_norm = d.norm()
            if d_norm > 1e-10 and w_norm > 1e-10:
                direction[k] = d * (w_norm / d_norm)
            else:
                direction[k] = d
        else:
            direction[k] = torch.zeros_like(v)
    return direction


def plot_loss_landscape(model, batch, steps=20, distance=0.5, title="Loss Landscape", save_path=None):
    print(f"Computing loss landscape ({steps}x{steps} = {steps**2} evaluations)...")

    model.eval()
    center = {k: v.clone() for k, v in model.state_dict().items()}
    dir1 = random_direction(center)
    dir2 = random_direction(center)

    alphas = np.linspace(-distance, distance, steps)
    betas = np.linspace(-distance, distance, steps)
    losses = np.zeros((steps, steps))

    batch = {k: v for k, v in batch.items() if k != "embed"}

    for i, alpha in enumerate(alphas):
        for j, beta in enumerate(betas):
            perturbed = {
                k: center[k] + alpha * dir1[k] + beta * dir2[k]
                for k in center
            }
            model.load_state_dict(perturbed)
            with torch.no_grad():
                try:
                    out = model(**batch)
                    losses[i, j] = out["loss"].item()
                except Exception:
                    losses[i, j] = float('nan')
        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{steps} rows done...")

    model.load_state_dict(center)
    model.eval()
    print("Done. Plotting...")

    fig = plt.figure(figsize=(18, 6))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    A, B = np.meshgrid(alphas, betas)
    Z = np.nan_to_num(losses.T, nan=np.nanmax(losses))

    ax1 = fig.add_subplot(131, projection='3d')
    ax1.plot_surface(A, B, Z, cmap='plasma', alpha=0.85, linewidth=0)
    ax1.set_xlabel('Direction 1', fontsize=9)
    ax1.set_ylabel('Direction 2', fontsize=9)
    ax1.set_zlabel('Loss', fontsize=9)
    ax1.set_title('3D Loss Surface', fontsize=10)
    ax1.view_init(elev=30, azim=-60)

    ax2 = fig.add_subplot(132)
    contour = ax2.contourf(A, B, Z, levels=30, cmap='plasma')
    plt.colorbar(contour, ax=ax2)
    ax2.scatter([0], [0], color='cyan', s=120, zorder=5, label='Current weights')
    ax2.set_xlabel('Direction 1', fontsize=9)
    ax2.set_ylabel('Direction 2', fontsize=9)
    ax2.set_title('Top-down Contour', fontsize=10)
    ax2.legend(fontsize=8)

    ax3 = fig.add_subplot(133)
    mid = steps // 2
    ax3.plot(alphas, losses[:, mid], 'b-', linewidth=2, label='Direction 1')
    ax3.plot(betas, losses[mid, :], 'r-', linewidth=2, label='Direction 2')
    ax3.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    current_loss = losses[mid, mid]
    ax3.axhline(y=current_loss, color='gray', linestyle='--', alpha=0.5,
                label=f'Current loss: {current_loss:.4f}')
    ax3.set_xlabel('Perturbation magnitude', fontsize=9)
    ax3.set_ylabel('Loss', fontsize=9)
    ax3.set_title('1D Cross-sections', fontsize=10)
    ax3.legend(fontsize=7)

    valid = losses[~np.isnan(losses)]
    sharpness = valid.max() - valid.min()
    if sharpness < 0.1:
        basin_type = "Wide smooth basin -> stable training"
    elif sharpness < 0.5:
        basin_type = "Moderate basin -> some oscillation risk"
    else:
        basin_type = "Sharp narrow basin -> high oscillation risk"

    stats = (
        f"Run: LMM standalone + codemap + batch=256\n"
        f"Config: lr=1e-4, weight_decay=1e-4, max_grad_norm=1.0\n"
        f"Current loss:     {current_loss:.4f}\n"
        f"Min in landscape: {valid.min():.4f}\n"
        f"Max in landscape: {valid.max():.4f}\n"
        f"Sharpness:        {sharpness:.4f}\n"
        f"Basin type:       {basin_type}"
    )
    fig.text(0.01, 0.02, stats, fontsize=8, family='monospace',
             verticalalignment='bottom',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.tight_layout(rect=[0, 0.2, 1, 0.95])

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")

    plt.show()
    print(f"\nSharpness: {sharpness:.4f} -- {basin_type}")


torch.manual_seed(42)
test_batch_landscape = next(iter(test_dataloader))

plot_loss_landscape(
    model=lmm_model,
    batch=test_batch_landscape,
    steps=20,
    distance=0.5,
    title=(
        "Loss Landscape: LMM Standalone + code_mapping + batch=256\n"
        "3 fixes: lr=1e-4, weight_decay=1e-4, max_grad_norm=1.0 | MIMIC-III"
    ),
    save_path="loss_landscape_lmm_standalone_batch256.png",
)